# Layer Sweep — % Significant Neurons vs Layer
Identifies the best layer per model for use in downstream notebooks.
Run at pc=20 (fixed), all layers available in EmbedCache.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
# ctx200 models — all 8 new models with 200-word causal sliding window
MODELS = {
    'llama-2-7b_ctx200':      {'n_layers': 33, 'color': '#d6604d', 'marker': 'o'},
    'llama-3.1-8b_ctx200':    {'n_layers': 33, 'color': '#b2182b', 'marker': 's'},
    'gpt2-xl_ctx200':         {'n_layers': 49, 'color': '#762a83', 'marker': 'P'},
    'opt-350m_ctx200':        {'n_layers': 24, 'color': '#9970ab', 'marker': 'X'},
    'gpt2-medium_ctx200':     {'n_layers': 25, 'color': '#4dac26', 'marker': '^'},
    'gpt2_ctx200':            {'n_layers': 13, 'color': '#2166ac', 'marker': 'D'},
    'bert-base_ctx200':       {'n_layers': 13, 'color': '#f4a582', 'marker': 'v'},
    'bert-base-causal_ctx200':{'n_layers': 13, 'color': '#d1e5f0', 'marker': '<'},
}

N_COMPONENTS = 20
REGION       = 'hippocampus'
CONDITIONS   = ['self', 'other']

GLM_BASE  = '/scratch/aniluchavez/ConvoDATAS/SemanticGLM'
SCRIPT    = '/scratch/aniluchavez/hippocampal-speaker-semantics/scripts/semantic_glm.py'
CONDA_ENV = 'gpt2_embed'
PROJECT   = '/scratch/aniluchavez/hippocampal-speaker-semantics'

In [ ]:
import os, pickle, subprocess, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 130})

def result_path(model, layer):
    return Path(GLM_BASE) / model / f'pc{N_COMPONENTS}' / f'L{layer:02d}_all.pkl'

def load_result(model, layer):
    p = result_path(model, layer)
    if not p.exists():
        return None
    obj = pickle.load(open(p, 'rb'))
    df = obj if isinstance(obj, pd.DataFrame) else obj.get('df', obj)
    # Require at least 10 patients worth of data
    if df[(df['region'] == REGION)]['patient'].nunique() < 10:
        return None
    return df

def run_layer(model, layer):
    p = result_path(model, layer)
    if p.exists():
        return
    # Strip _ctx200 suffix to get the base model tag; pass as --context_tag
    if model.endswith('_ctx200'):
        base_model = model[:-len('_ctx200')]
        ctx_arg = '--context_tag _ctx200'
    else:
        base_model = model
        ctx_arg = ''
    print(f'  {model} L{layer}: running...', flush=True)
    cmd = (f'conda run -n {CONDA_ENV} python3 -u {SCRIPT} '
           f'--layer {layer} --n_components {N_COMPONENTS} --model {base_model} {ctx_arg}')
    r = subprocess.run(cmd, shell=True, cwd=PROJECT, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  FAILED ({model} L{layer}):', r.stderr[-500:])
    else:
        print(f'  {model} L{layer}: done')

In [ ]:
# Run any missing layers (skips already-cached)
for model, cfg in MODELS.items():
    for L in range(cfg['n_layers']):
        run_layer(model, L)

In [ ]:
# Collect % significant and median R² per layer per model
rows = []
best_layers = {}

for model, cfg in MODELS.items():
    for layer in range(cfg['n_layers']):
        df = load_result(model, layer)
        if df is None:
            continue
        for cond in CONDITIONS:
            sub = df[(df['region'] == REGION) & (df['condition'] == cond)]
            if sub.empty:
                continue
            n_sig = sub['significant'].sum()
            n_tot = len(sub)
            sig_r2 = sub.loc[sub['significant'], 'r2']
            rows.append({
                'model':       model,
                'layer':       layer,
                'condition':   cond,
                'pct_sig':     100 * n_sig / n_tot,
                'n_sig':       n_sig,
                'n_tot':       n_tot,
                'med_r2_sig':  sig_r2.median() if n_sig else np.nan,
            })

stats = pd.DataFrame(rows)

# Find best layer per model (max mean pct_sig across conditions)
if not stats.empty:
    mean_pct = stats.groupby(['model', 'layer'])['pct_sig'].mean().reset_index()
    for model in MODELS:
        sub = mean_pct[mean_pct['model'] == model]
        if sub.empty:
            best_layers[model] = None
        else:
            best_layers[model] = int(sub.loc[sub['pct_sig'].idxmax(), 'layer'])
    print('Best layers per model:')
    for m, L in best_layers.items():
        if L is not None:
            mean_pct_at_L = mean_pct[(mean_pct['model']==m) & (mean_pct['layer']==L)]['pct_sig'].values[0]
            print(f'  {m:20s}  L{L:02d}  ({mean_pct_at_L:.1f}% sig)')
else:
    print('No results yet — run cell above first')

In [ ]:
# ── PLOT 1: % significant neurons vs layer ──────────────────────────────────
if stats.empty:
    print('No data yet')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax_i, cond in enumerate(CONDITIONS):
        ax = axes[ax_i]
        for model, cfg in MODELS.items():
            sub = stats[(stats['model'] == model) & (stats['condition'] == cond)]
            if sub.empty:
                continue
            # Normalize x-axis: layer / (n_layers-1) → 0..1 for cross-model comparison
            x = sub['layer'].values / (cfg['n_layers'] - 1)
            ax.plot(sub['layer'], sub['pct_sig'],
                    color=cfg['color'], marker=cfg['marker'],
                    markersize=5, linewidth=1.8, alpha=0.85,
                    label=model)
            # Mark best layer
            best_L = best_layers.get(model)
            if best_L is not None:
                best_row = sub[sub['layer'] == best_L]
                if not best_row.empty:
                    ax.scatter(best_L, best_row['pct_sig'].values[0],
                               color=cfg['color'], s=120, zorder=5,
                               edgecolors='k', linewidths=1.2)
        ax.set_xlabel('Layer index')
        ax.set_ylabel('% significant neurons')
        ax.set_title(f'{REGION} / {cond}')
        ax.legend(fontsize=8.5, frameon=False)
    plt.suptitle(f'Layer sweep — pc={N_COMPONENTS}  (filled circle = best layer)', y=1.01)
    plt.tight_layout()
    plt.savefig('../figures/00_layer_sweep_pct_sig.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── PLOT 2: Median R² (sig neurons) vs layer ──────────────────────────────
if stats.empty:
    print('No data yet')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax_i, cond in enumerate(CONDITIONS):
        ax = axes[ax_i]
        for model, cfg in MODELS.items():
            sub = stats[(stats['model'] == model) & (stats['condition'] == cond)].dropna(subset=['med_r2_sig'])
            if sub.empty:
                continue
            ax.plot(sub['layer'], sub['med_r2_sig'],
                    color=cfg['color'], marker=cfg['marker'],
                    markersize=5, linewidth=1.8, alpha=0.85,
                    label=model)
        ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
        ax.set_xlabel('Layer index')
        ax.set_ylabel('Median deviance R² (sig neurons)')
        ax.set_title(f'{REGION} / {cond}')
        ax.legend(fontsize=8.5, frameon=False)
    plt.suptitle(f'Effect size by layer — pc={N_COMPONENTS}', y=1.01)
    plt.tight_layout()
    plt.savefig('../figures/00_layer_sweep_r2.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# ── PLOT 3: Model comparison bar chart at best layer per model ────────────
if stats.empty or not any(v is not None for v in best_layers.values()):
    print('No data yet')
else:
    bar_rows = []
    for model, best_L in best_layers.items():
        if best_L is None:
            continue
        df = load_result(model, best_L)
        if df is None:
            continue
        for cond in CONDITIONS:
            sub = df[(df['region'] == REGION) & (df['condition'] == cond)]
            if sub.empty:
                continue
            bar_rows.append({
                'model':      model,
                'condition':  cond,
                'best_layer': best_L,
                'pct_sig':    100 * sub['significant'].mean(),
                'med_r2_sig': sub.loc[sub['significant'], 'r2'].median() if sub['significant'].any() else np.nan,
            })

    bar_df = pd.DataFrame(bar_rows)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax_i, cond in enumerate(CONDITIONS):
        ax = axes[ax_i]
        sub = bar_df[bar_df['condition'] == cond]
        colors = [MODELS[m]['color'] for m in sub['model']]
        bars = ax.bar(range(len(sub)), sub['pct_sig'], color=colors, alpha=0.8, width=0.6)
        for i, (_, row) in enumerate(sub.iterrows()):
            ax.text(i, row['pct_sig'] + 0.3,
                    f"L{row['best_layer']}\nR²={row['med_r2_sig']:.4f}",
                    ha='center', fontsize=7.5, color='#333')
        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels(sub['model'], rotation=20, ha='right', fontsize=9)
        ax.set_ylabel('% significant neurons (hippocampus)')
        ax.set_title(f'{cond}')

    plt.suptitle(f'Model comparison @ best layer  (pc={N_COMPONENTS})', y=1.01)
    plt.tight_layout()
    plt.savefig('../figures/00_model_comparison_best_layer.pdf', bbox_inches='tight')
    plt.show()
    print('\nBest layers used:')
    for _, r in bar_df[bar_df['condition']=='other'].iterrows():
        print(f"  {r['model']:20s}  best_layer=L{r['best_layer']:02d}  {r['pct_sig']:.1f}% sig")